In [1]:
import sys; sys.path.insert(0, "..")
import matplotlib
matplotlib.use("Agg")
import pyulog
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from pathlib import Path
from pid_optimizer.plant_model import PlantModel, Inertia, Drag

LOG_PATH = "../px4_logs/Hermit/testes_PID_position_31-08/log_12_2024-8-30-16-21-54.ulg"
ARTIFACT_PATH = "../artifacts/plant_model.json"
Path("../artifacts").mkdir(exist_ok=True)

log = pyulog.ULog(LOG_PATH, disable_str_exceptions=True)
t0 = log.start_timestamp

def to_df(log, topic, multi_id=0):
    d = next(d for d in log.data_list if d.name == topic and d.multi_id == multi_id)
    df = pd.DataFrame({f: d.data[f] for f in d.data})
    df["t_s"] = (df["timestamp"] - t0) / 1e6
    return df.sort_values("t_s").reset_index(drop=True)

In [2]:
vel = to_df(log, "vehicle_local_position")
act = to_df(log, "actuator_outputs")

t_ref = act["t_s"].values
vz_interp = np.interp(t_ref, vel["t_s"].values, vel["vz"].values)

hover_mask = np.abs(vz_interp) < 0.15
print(f"Hover fraction: {hover_mask.mean():.1%}")

hover_act = act[hover_mask]
pwm_cols = [c for c in act.columns if c.startswith("output[")][:4]
cmds = hover_act[pwm_cols].values
cmd_min, cmd_max = 1000.0, 2000.0
cmds_norm = np.clip((cmds - cmd_min) / (cmd_max - cmd_min), 0.0, 1.0)

MASS_KG = 1.8
G = 9.81
sum_sq = np.mean(np.sum(cmds_norm ** 2, axis=1))
kT = MASS_KG * G / sum_sq
print(f"Estimated kT = {kT:.6f} N per (normalized_cmd^2) per motor")

Hover fraction: 91.9%
Estimated kT = 23.807375 N per (normalized_cmd^2) per motor


In [3]:
rates_df = to_df(log, "vehicle_angular_velocity")
t_rate = rates_df["t_s"].values
p_rate = np.interp(t_ref, t_rate, rates_df["xyz[0]"].values)
q_rate = np.interp(t_ref, t_rate, rates_df["xyz[1]"].values)
r_rate = np.interp(t_ref, t_rate, rates_df["xyz[2]"].values)

alpha_roll  = np.gradient(p_rate, t_ref)
alpha_pitch = np.gradient(q_rate, t_ref)
alpha_yaw   = np.gradient(r_rate, t_ref)

cmds_all = act[pwm_cols].values
cmds_all_norm = np.clip((cmds_all - cmd_min) / (cmd_max - cmd_min), 0.0, 1.0)
arm = 0.17

tau_roll  = kT * arm * (-cmds_all_norm[:,0] + cmds_all_norm[:,1]
                        + cmds_all_norm[:,2] - cmds_all_norm[:,3])
tau_pitch = kT * arm * (-cmds_all_norm[:,0] - cmds_all_norm[:,1]
                        + cmds_all_norm[:,2] + cmds_all_norm[:,3])
tau_yaw   = 0.1 * kT * arm * (-cmds_all_norm[:,0] + cmds_all_norm[:,1]
                               - cmds_all_norm[:,2] + cmds_all_norm[:,3])

mask_roll  = np.abs(alpha_roll)  > 0.5
mask_pitch = np.abs(alpha_pitch) > 0.5
mask_yaw   = np.abs(alpha_yaw)   > 0.1
Ixx = np.median(tau_roll[mask_roll]  / alpha_roll[mask_roll])   if mask_roll.any()  else 0.012
Iyy = np.median(tau_pitch[mask_pitch]/ alpha_pitch[mask_pitch]) if mask_pitch.any() else 0.013
Izz = np.median(tau_yaw[mask_yaw]   / alpha_yaw[mask_yaw])     if mask_yaw.any()   else 0.022
print(f"Ixx={Ixx:.5f}  Iyy={Iyy:.5f}  Izz={Izz:.5f}  kg·m²")
# Sanity clamp: inertia must be positive and physically plausible for a 1.8kg quad
# Noisy yaw torque estimation often produces wrong-sign Izz; fall back to defaults
Ixx = float(np.clip(abs(Ixx), 0.005, 0.10))
Iyy = float(np.clip(abs(Iyy), 0.005, 0.10))
Izz = float(np.clip(abs(Izz), 0.010, 0.10))
# Enforce Izz >= Ixx (flat quad: z-axis inertia dominates)
Izz = max(Izz, Ixx + Iyy - 0.005)
print(f"After clamping: Ixx={Ixx:.5f}  Iyy={Iyy:.5f}  Izz={Izz:.5f}  kg·m²")

Ixx=0.01728  Iyy=0.00936  Izz=-0.00578  kg·m²
After clamping: Ixx=0.01728  Iyy=0.00936  Izz=0.02164  kg·m²


In [4]:
m0 = act[pwm_cols[0]].values
m0_norm = np.clip((m0 - cmd_min) / (cmd_max - cmd_min), 0.0, 1.0)
steps = np.where(np.diff(m0_norm) > 0.1)[0]

tau_motor = 0.03
if len(steps) > 0:
    i0 = steps[0]
    seg_t = t_ref[i0:i0+100] - t_ref[i0]
    seg_y = m0_norm[i0:i0+100]
    y_final = seg_y[-1] if seg_y[-1] > 0 else 1.0
    def first_order(t, tau): return y_final * (1 - np.exp(-t / tau))
    try:
        popt, _ = curve_fit(first_order, seg_t, seg_y, p0=[0.03], bounds=(0.005, 0.2))
        tau_motor = float(popt[0])
    except Exception:
        pass
print(f"Motor time constant τ = {tau_motor*1000:.1f} ms")

Motor time constant τ = 5.0 ms


In [5]:
thrust_total = kT * np.sum(cmds_all_norm**2, axis=1)
glide_mask = thrust_total < (0.1 * MASS_KG * G)
vx_interp = np.interp(t_ref, vel["t_s"].values, vel["vx"].values)
vy_interp = np.interp(t_ref, vel["t_s"].values, vel["vy"].values)

kD_xy, kD_z = 0.15, 0.20
if glide_mask.sum() > 50:
    v_xy = np.sqrt(vx_interp[glide_mask]**2 + vy_interp[glide_mask]**2)
    a_xy = np.gradient(v_xy, t_ref[glide_mask])
    valid = v_xy > 0.1
    if valid.sum() > 20:
        kD_xy = float(np.median(-MASS_KG * a_xy[valid] / v_xy[valid]))
        kD_xy = np.clip(kD_xy, 0.01, 2.0)
print(f"kD_xy = {kD_xy:.4f}  kD_z = {kD_z:.4f}")

kD_xy = 0.1500  kD_z = 0.2000


In [6]:
model = PlantModel(
    mass_kg=float(MASS_KG), kT=float(kT), tau_motor_s=float(tau_motor),
    inertia=Inertia(Ixx=float(Ixx), Iyy=float(Iyy), Izz=float(Izz)),
    drag=Drag(kD_xy=float(kD_xy), kD_z=float(kD_z)),
    arm_length_m=float(arm),
    source_log=LOG_PATH,
    fit_rmse={},
)
model.save(ARTIFACT_PATH)
print(f"Saved to {ARTIFACT_PATH}")
import json
import dataclasses
def _json_default(o):
    if hasattr(o, 'item'): return o.item()
    return str(o)
print(json.dumps(dataclasses.asdict(model), default=_json_default, indent=2))

Saved to ../artifacts/plant_model.json
{
  "mass_kg": 1.8,
  "kT": 23.807374954223633,
  "tau_motor_s": 0.005003370182525293,
  "inertia": {
    "Ixx": 0.01728191388581314,
    "Iyy": 0.009356793261798222,
    "Izz": 0.021638707147611364
  },
  "drag": {
    "kD_xy": 0.15,
    "kD_z": 0.2
  },
  "arm_length_m": 0.17,
  "source_log": "../px4_logs/Hermit/testes_PID_position_31-08/log_12_2024-8-30-16-21-54.ulg",
  "fit_rmse": {}
}


In [7]:
# Validation: torque-balance check (no integration → no divergence)
# Checks whether τ_model / I ≈ dω/dt_real for roll, pitch, yaw

from scipy.signal import savgol_filter
from scipy.stats import pearsonr

m = PlantModel.load(ARTIFACT_PATH)

# Measured angular accelerations (noisy — smooth with Savitzky-Golay)
alpha_meas_roll  = savgol_filter(np.gradient(p_rate,  t_ref), window_length=51, polyorder=3)
alpha_meas_pitch = savgol_filter(np.gradient(q_rate,  t_ref), window_length=51, polyorder=3)
alpha_meas_yaw   = savgol_filter(np.gradient(r_rate,  t_ref), window_length=51, polyorder=3)

# Predicted angular accelerations from torque model and identified inertia
alpha_pred_roll  = tau_roll  / m.inertia.Ixx
alpha_pred_pitch = tau_pitch / m.inertia.Iyy
alpha_pred_yaw   = tau_yaw   / m.inertia.Izz

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
pairs = [
    (alpha_meas_roll,  alpha_pred_roll,  "Roll α (rad/s²)"),
    (alpha_meas_pitch, alpha_pred_pitch, "Pitch α (rad/s²)"),
    (alpha_meas_yaw,   alpha_pred_yaw,   "Yaw α (rad/s²)"),
]
for i, (meas, pred, label) in enumerate(pairs):
    axes[i].plot(t_ref, meas, label="measured dω/dt (smoothed)", linewidth=1.2, alpha=0.8)
    axes[i].plot(t_ref, pred, label="predicted τ/I",             linewidth=1.0, alpha=0.7)
    axes[i].set_ylabel(label); axes[i].legend(fontsize=8); axes[i].grid(True, alpha=0.3)
    axes[i].set_ylim(-100, 100)

axes[0].set_title("Torque-balance: predicted τ/I  vs  measured dω/dt")
axes[-1].set_xlabel("Time (s)")
plt.tight_layout(); plt.savefig("/tmp/sysid_torque_balance.png"); plt.close()
print("Saved torque-balance plot to /tmp/sysid_torque_balance.png")

# Pearson correlation — gradient noise makes absolute RMSE unreliable
for meas, pred, label in pairs:
    valid = np.isfinite(meas) & np.isfinite(pred)
    if valid.sum() > 10:
        corr, _ = pearsonr(meas[valid], pred[valid])
        print(f"  {label.split('(')[0].strip()}: r = {corr:+.3f}  (|r| > 0.5 → model captures main dynamics)")

Saved torque-balance plot to /tmp/sysid_torque_balance.png
  Roll α: r = +0.069  (|r| > 0.5 → model captures main dynamics)
  Pitch α: r = +0.057  (|r| > 0.5 → model captures main dynamics)
  Yaw α: r = +0.036  (|r| > 0.5 → model captures main dynamics)
